# 05 · Un parque real y conclusiones

**Clases 1, 4 y 12:** transformar la información en decisiones y leer residuos.

Comparar lo observado con lo estimado para el clima de cada mes señala parques para revisar. Un desvío no confirma la causa: CAMMESA no publica fallas etiquetadas y no se arma una matriz de confusión.

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    import subprocess
    root = Path('/content/tpo-solar')
    if not root.exists():
        subprocess.run(['git','clone','https://github.com/imelinc/tpo-solar.git',str(root)],check=True)
    os.chdir(root)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, DATA_PROCESSED
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
A = pd.read_parquet(DATA_PROCESSED / 'defensa_argentina.parquet')
E = json.loads((DATA_PROCESSED / 'defensa_evidencia.json').read_text())
caso = A[A['central'].eq(E['ejemplo_parque'])].sort_values('fecha')
print(caso['central_descripcion'].iloc[0], caso['provincia'].iloc[0], caso['potencia_mw'].iloc[0], 'MW')
fig = go.Figure()
fig.add_scatter(x=caso['fecha'], y=caso['factor_capacidad'], name='Observado')
fig.add_scatter(x=caso['fecha'], y=caso['pred'], name='Estimado', line=dict(dash='dot'))
fig.update_layout(title='Clase 12 · Observado contra estimado', yaxis_title='Factor de capacidad')
fig.show()
print('Meses en cero:', caso.loc[caso['factor_capacidad'].eq(0), 'fecha'].tolist())

## Qué parques se alejan más en la prueba

El error absoluto medio por parque en la prueba ordena dónde mirar primero. Es una lista para revisar, no un diagnóstico.

In [ ]:
prueba = A[A['particion'].eq('prueba')].assign(error=lambda x: (x['factor_capacidad'] - x['pred']).abs(),
                                                 residuo=lambda x: x['factor_capacidad'] - x['pred'])
ranking = prueba.groupby(['central','central_descripcion','provincia']).agg(meses=('fecha','size'), mae=('error','mean'), residuo_medio=('residuo','mean'))
ranking.sort_values('mae', ascending=False).head(10).round(3)

## Cierre

- Con ubicación y clima, el modelo estima el factor mensual con un error medio cercano a 0,04 en la prueba retrospectiva.
- Explica cerca de la mitad de la variación (R² de prueba). El clima de un sitio nuevo se copia del parque más cercano; a más de 150 km la copia es débil.
- Fuera del rango de los parques de entrenamiento, un modelo de árboles no extrapola: la aplicación marca esos puntos como caso límite.
- La energía estimada no es producción medida, y un desvío no prueba la causa.